# Swift: Neural Video Streaming Pipeline
This notebook orchestrates the end-to-end workflow for the Swift codec, including hardware profiling, two-stage training, and evaluation.

## 1. Environment Setup
Ensure all dependencies are installed and directories are ready.

In [ ]:
import os
import sys

# Create necessary directories
os.makedirs("models", exist_ok=True)
os.makedirs("outputs/evaluation", exist_ok=True)
os.makedirs("configs", exist_ok=True)

print("Project root:", os.getcwd())
print("Python version:", sys.version)

## 2. Hardware Profiling
Calibrate the ABR policy by measuring local GPU decoding latency for different exit resolutions.

In [ ]:
!python scripts/profile_hardware.py

## 3. Training Stage 1: Multi-Level Autoencoder
Train the motion-aware residual encoder on the triplet dataset.

In [ ]:
# Note: Training might take a long time. 
# You can reduce --epochs for testing purposes.
!python codec/autoencoder/train_script.py --train-dir data/original_frames --epochs 20 --batch-size 4

## 4. Training Stage 2: Singleshot Decoder
Train the early-exit decoder using frozen bitstreams from the Stage 1 Autoencoder.

In [ ]:
!python codec/singleshot/train_singleshot.py

## 5. End-to-End Evaluation
Run the full evaluation suite including PSNR, SSIM, Latency, and Baseline comparisons.

In [ ]:
!python scripts/evaluate.py

## 6. Visualize Results
Show the generated Rate-Distortion curves and error heatmaps.

In [ ]:
from IPython.display import Image, display

print("\n--- RD-Curve Comparison ---")
rd_curve_path = "outputs/evaluation/rd_curve_comparison.png"
if os.path.exists(rd_curve_path):
    display(Image(filename=rd_curve_path))
else:
    print("RD Curve not found. Ensure evaluate.py ran successfully.")

print("\n--- Sample Pixel Error Heatmap ---")
heatmap_dir = "outputs/evaluation/heatmaps"
if os.path.exists(heatmap_dir):
    heatmaps = [f for f in os.listdir(heatmap_dir) if f.endswith('.png')]
    if heatmaps:
        display(Image(filename=os.path.join(heatmap_dir, heatmaps[0])))
else:
    print("Heatmaps not found.")